In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer

In [2]:
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.features import CATEGORICAL_COLS, prep_one_split, DROP_COLS, NUMERICAL_COLS

2026-06-14 11:34:52.415 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [3]:
from credit_risk.evaluation import evaluate_model

In [4]:
train_df, val_df, test_df, _ = load_splits(path=AFTER_EDA)

2026-06-14 11:34:53.102 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-14 11:34:53.113 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-14 11:34:53.390 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
# regime 2 dropping int_rate, grade and sub_grade
CATEGORICAL_COLS.remove('grade')
CATEGORICAL_COLS.remove('sub_grade')
NUMERICAL_COLS.remove('int_rate')
DROP_COLS.remove('zip_code')

In [6]:
TARGET_ENC = ['zip_code']

In [7]:
filtered_train_df, y_train = prep_one_split(df=train_df, drop_cols=DROP_COLS)
filtered_val_df, y_val = prep_one_split(df=val_df, drop_cols=DROP_COLS)
filtered_test_df, y_test = prep_one_split(df=test_df, drop_cols=DROP_COLS)

2026-06-14 11:34:53.917 | INFO     | credit_risk.features:prep_one_split:207 - Inside Function: prep_one_split
2026-06-14 11:34:53.918 | INFO     | credit_risk.features:split_target_and_features:143 - Inside Function: split_target_and_features
2026-06-14 11:34:53.918 | INFO     | credit_risk.features:split_target_and_features:144 - Splitting the target and the features...
2026-06-14 11:34:53.920 | INFO     | credit_risk.features:split_target_and_features:147 - features and target are splitted successfully...
2026-06-14 11:34:53.920 | INFO     | credit_risk.features:add_credit_yrs:160 - Inside Function: add_credit_yrs
2026-06-14 11:34:53.920 | INFO     | credit_risk.features:add_credit_yrs:162 - Adding the feature 'credit_yrs'
2026-06-14 11:34:53.931 | INFO     | credit_risk.features:add_credit_yrs:164 - 'credit_age_yrs added successfully!'
2026-06-14 11:34:53.931 | INFO     | credit_risk.features:add_fico_mid:169 - Inside Function: add_fico_mid
2026-06-14 11:34:53.931 | INFO     | cred

In [8]:
num_pipeline = Pipeline([
    ('inpute', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

tar_pipeline = Pipeline([
    ('target_enc', TargetEncoder(target_type='binary')),
    ('scalar', StandardScaler())
])

In [9]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERICAL_COLS),
    ('cat', cat_pipeline, CATEGORICAL_COLS),
    ('tar', tar_pipeline, TARGET_ENC)
])

In [10]:
X_train_feat = preprocessor.fit_transform(X=filtered_train_df, y=y_train.to_numpy())

In [11]:
X_val_feat = preprocessor.transform(filtered_val_df)
X_test_feat = preprocessor.transform(filtered_test_df)

In [12]:
from credit_risk.utils import to_jsonable, train

In [13]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='LR'
)

In [15]:
lr_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [16]:
lr_eval_dict

{'threshold': np.float64(0.15000000000000002),
 'train': {'ROC-AUC': 0.6959398667307165,
  'PR-AUC': 0.30771038910148235,
  'brier_score': 0.12910203541382406,
  'precision': 0.2495918795429051,
  'recall': 0.705393388749275,
  'confusion_matrix': array([[223884, 164563],
         [ 22860,  54735]])},
 'val': {'ROC-AUC': 0.7003612669679651,
  'PR-AUC': 0.33849261345353476,
  'brier_score': 0.1382497577058737,
  'precision': 0.2698383039885016,
  'recall': 0.7299094125422806,
  'confusion_matrix': array([[190638, 152403],
         [ 20841,  56322]])},
 'test': {'ROC-AUC': 0.6870712536995723,
  'PR-AUC': 0.30043669199507395,
  'brier_score': 0.13154306095573204,
  'precision': 0.2506223609142319,
  'recall': 0.6873128382647875,
  'confusion_matrix': array([[209306, 149608],
         [ 22763,  50035]])}}

In [17]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='RF'
)

In [18]:
rf_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [19]:
rf_eval_dict

{'threshold': np.float64(0.18000000000000002),
 'train': {'ROC-AUC': 1.0,
  'PR-AUC': 1.0000000000000002,
  'brier_score': 0.01830930989052489,
  'precision': 0.9657726056381853,
  'recall': 1.0,
  'confusion_matrix': array([[385697,   2750],
         [     0,  77595]])},
 'val': {'ROC-AUC': 0.6817787683329724,
  'PR-AUC': 0.3151791450670715,
  'brier_score': 0.14044992932004455,
  'precision': 0.2631198503518977,
  'recall': 0.6981714033928178,
  'confusion_matrix': array([[192167, 150874],
         [ 23290,  53873]])},
 'test': {'ROC-AUC': 0.660118232310421,
  'PR-AUC': 0.27480465584252795,
  'brier_score': 0.1342453279037877,
  'precision': 0.23798304202080056,
  'recall': 0.6465699607132064,
  'confusion_matrix': array([[208200, 150714],
         [ 25729,  47069]])}}

In [20]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='XGB'
)

In [21]:
xgb_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [22]:
xgb_eval_dict

{'threshold': np.float64(0.15000000000000002),
 'train': {'ROC-AUC': 0.7626480042974798,
  'PR-AUC': 0.4162187480771513,
  'brier_score': 0.11978831887245178,
  'precision': 0.2805695416969158,
  'recall': 0.7719956182743734,
  'confusion_matrix': array([[234845, 153602],
         [ 17692,  59903]])},
 'val': {'ROC-AUC': 0.7023485083756467,
  'PR-AUC': 0.33594078369153113,
  'brier_score': 0.13855069875717163,
  'precision': 0.2732368373261869,
  'recall': 0.7214468074076954,
  'confusion_matrix': array([[194971, 148070],
         [ 21494,  55669]])},
 'test': {'ROC-AUC': 0.6866456962212546,
  'PR-AUC': 0.29838897521591234,
  'brier_score': 0.13225309550762177,
  'precision': 0.25220688065168945,
  'recall': 0.6762136322426441,
  'confusion_matrix': array([[212956, 145958],
         [ 23571,  49227]])}}

In [23]:
train_zips = set(filtered_train_df['zip_code'].dropna().unique())
val_zips = set(filtered_val_df['zip_code'].dropna().unique())
test_zips = set(filtered_test_df['zip_code'].dropna().unique())
print(f"val zips not in train: {len(val_zips - train_zips)} / {len(val_zips)}")
print(f"test zips not in train: {len(test_zips - train_zips)} / {len(test_zips)}")

val zips not in train: 47 / 914
test zips not in train: 42 / 911
